# Notebook 4 — Data Quality Checks

**Goals**

1. Identify *every* common data-quality issue in a forecasting dataset
   **before** doing any feature engineering or modelling.
2. Distinguish **missingness in values** from **gaps in time** — they are
   different problems and need different fixes.
3. Quantify and visualise outliers with three complementary detectors.
4. Catch hidden data-integrity bugs: duplicates, sign/zero anomalies, and
   inconsistent static features.

>  **Why this notebook matters.** A clean dataset is worth more than a
> sophisticated model. Forecasting models are particularly sensitive to
> "phantom" zeros (an unrecorded date assumed to mean zero demand) and to
> point outliers that distort lag/rolling features later on.

The toolkit uses `spec` (declared in the dataset block below) so every
function works generically on any forecasting dataset.


In [1]:
# ──────────────────────────────────────────────────────────────────────────
# COLAB SETUP — run this once at the top of every tutorial notebook.
# It installs plotly + statsmodels and makes the toolkit importable.
# If you are running locally (not in Colab) the !pip line is harmless.
# ──────────────────────────────────────────────────────────────────────────
!pip install -q plotly statsmodels
import sys, os
# If you uploaded forecasting_toolkit.zip to Colab, unzip it once:
#   !unzip -o forecasting_toolkit.zip
# Otherwise place forecasting_toolkit/ next to this notebook.
sys.path.insert(0, os.path.abspath('.'))

import forecasting_toolkit as ft
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'colab'   # change to 'notebook' for local Jupyter


In [2]:
# ──────────────────────────────────────────────────────────────────────────
# POINT THIS AT YOUR DATASET — fill in the four lines below.
# Everything in this notebook works for ANY tabular sales/demand dataset
# (M5, Rossmann, Walmart Store Item Demand, custom CSVs, etc.).
#
#   DATA_PATH    – path to your CSV / Parquet file
#   DATE_COL     – name of the timestamp column
#   TARGET_COL   – name of the column you want to forecast
#   KEY_COLS     – list of columns that together identify ONE time series
#   STATIC_COLS  – columns constant within a key (e.g. store_type, category)
#   DYNAMIC_COLS – columns that vary in time within a key (e.g. promo, price)
#   FREQUENCY    – pandas offset alias: 'D','W','MS','H',…
# ──────────────────────────────────────────────────────────────────────────
DATA_PATH    = './datasets/rohlik_kaggle/sales_processed_tft.csv'
DATE_COL     = 'date'
TARGET_COL   = 'sales'
KEY_COLS     = ['unique_id']
STATIC_COLS  = ['warehouse','product_unique_id','name','L1_category_name_en','L2_category_name_en','L3_category_name_en','L4_category_name_en','country']
DYNAMIC_COLS = ['total_orders','sell_price_main','type_0_discount','type_1_discount','type_2_discount','type_3_discount','type_4_discount','type_5_discount','type_6_discount','holiday_name',
                'holiday','shops_closed','winter_school_holidays','school_holidays','weekday','week','month','day','is_month_start','is_month_end','quarter','weekend','days_since_2020']
FREQUENCY    = 'D'

from forecasting_toolkit import data_io
spec = data_io.make_spec(
    date_col=DATE_COL, target_col=TARGET_COL,
    key_cols=KEY_COLS, static_cols=STATIC_COLS,
    dynamic_cols=DYNAMIC_COLS, frequency=FREQUENCY,
)
df = data_io.load_data(DATA_PATH, spec)
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head()


Loaded 4,054,440 rows × 38 columns


,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,...,month,year,prev_year,day,is_month_start,is_month_end,quarter,weekend,days_since_2020,country
0,0,2022-07-18,Budapest_1,5289.0,3.97,710.89,0.09,0.0,0.0,0.00000,...,7,2022,2022,18,False,False,3,0,929,Hungary
1,0,2022-07-19,Budapest_1,5255.0,73.36,710.89,1.00,0.0,0.0,0.00000,...,7,2022,2022,19,False,False,3,0,930,Hungary
2,0,2022-07-20,Budapest_1,5334.0,558.09,710.89,0.96,0.0,0.0,0.45045,...,7,2022,2022,20,False,False,3,0,931,Hungary
3,0,2022-07-21,Budapest_1,5459.0,14.03,710.89,0.06,0.0,0.0,0.45045,...,7,2022,2022,21,False,False,3,0,932,Hungary
4,0,2022-07-22,Budapest_1,5461.0,558.53,710.89,0.97,0.0,0.0,0.45045,...,7,2022,2022,22,False,False,3,0,933,Hungary


In [3]:
import forecasting_toolkit as ft


## 4.1 Missing values — overall

`missing_value_report(df)` walks every column and reports how many rows
are NaN. This is the first lens to apply on any new dataset.

| Column        | What missingness usually means |
|---------------|--------------------------------|
| **target**    | Either truly absent or a recording gap. Rarely "zero". |
| **dynamic**   | Sensor / system glitch — needs imputation later.       |
| **static**    | Almost always a join issue — fix at source if possible. |
| **date**      | Drop or repair *before* anything else.                 |


In [4]:
mv_report = ft.quality.missing_value_report(df)
mv_report


,column,missing,missing_pct,non_null,dtype
0,holiday_name,3890663,95.96,163777,str
1,unique_id,0,0.00,4054440,int64
2,warehouse,0,0.00,4054440,str
3,total_orders,0,0.00,4054440,float64
4,sales,0,0.00,4054440,float64
5,date,0,0.00,4054440,datetime64[us]
6,sell_price_main,0,0.00,4054440,float64
7,availability,0,0.00,4054440,float64
8,type_1_discount,0,0.00,4054440,float64
9,type_0_discount,0,0.00,4054440,float64


A bar chart is faster than reading numbers when many columns are involved:


In [5]:
fig = ft.plotting.plot_missing_bar(mv_report,
        title='Missing values per column')
fig.show()


The **missingness heatmap** shows missing cells as dark stripes. Patterns
that span entire rows mean the row was bad; patterns that span entire
columns mean a join failed.


In [6]:
fig = ft.plotting.plot_missing_heatmap(df, max_cols=20)
fig.show()


## 4.2 Missing values — *per forecast key*

A column-level report can hide the fact that one column is mostly clean
**but a few series are 80% NaN**. `missing_by_series` slices missingness
by the spec's `key_cols`:


In [7]:
mv_series = ft.quality.missing_by_series(df, spec)
print(f'{len(mv_series):,} keys analysed.')
mv_series.head(10)


5,390 keys analysed.


,unique_id,n_obs,missing,missing_pct
0,5431,593,0,0.0
1,0,35,0,0.0
2,1,203,0,0.0
3,2,65,0,0.0
4,3,386,0,0.0
5,5,483,0,0.0
6,6,852,0,0.0
7,7,1063,0,0.0
8,8,714,0,0.0
9,9,1394,0,0.0


**Rule of thumb.** Series with > 50% missing target are usually safer to
drop or to forecast as a constant — they don't have enough signal to
support a custom model.


In [8]:
threshold_pct = 50
problem_keys = mv_series[mv_series['missing_pct'] > threshold_pct]
print(f'{len(problem_keys):,} keys are >{threshold_pct}% missing on the target.')
problem_keys.head()


0 keys are >50% missing on the target.


,unique_id,n_obs,missing,missing_pct


## 4.3 Time gaps — different from "missing values"!

A *time gap* is when a date is **entirely absent** from a series — there
is no row for that day at all. `missing_value_report` cannot see these
because the row isn't there to count as NaN.

Concretely:

| Situation        | Caught by `missing_value_report` | Caught by `detect_time_gaps` |
|------------------|----------------------------------|------------------------------|
| Row exists, target = NaN | ✅                          | ❌                            |
| Row missing entirely     | ❌                          | ✅                            |

`detect_time_gaps` reindexes each key to the spec's frequency and reports
how many timestamps are missing.


In [9]:
gaps = ft.quality.detect_time_gaps(df, spec)
print(f'{(gaps["missing_dates"] > 0).sum():,} keys have at least one time gap.')
gaps.head(10)


5,248 keys have at least one time gap.


,unique_id,n_obs,expected_obs,missing_dates,gap_pct,first_gap,last_gap
0,2017,58,1413,1355,95.90,2020-09-05,2024-06-10
1,5342,96,1404,1308,93.16,2020-08-21,2024-04-21
2,4776,39,1340,1301,97.09,2020-10-21,2024-05-23
3,2380,121,1414,1293,91.44,2020-09-29,2024-05-18
4,2382,136,1414,1278,90.38,2020-08-02,2024-06-09
5,1084,136,1396,1260,90.26,2020-08-02,2024-05-02
6,2379,170,1411,1241,87.95,2020-09-29,2024-06-11
7,3532,65,1302,1237,95.01,2020-11-18,2024-06-05
8,2381,174,1411,1237,87.67,2020-09-29,2024-05-18
9,3432,192,1416,1224,86.44,2020-08-20,2024-05-30


### Inspect actual missing dates for one key

`gap_examples` lists the actual missing timestamps for a single key, with
the weekday name. This is how you tell **structural gaps** (every Sunday
missing → store closed) from **random gaps** (sensor failure).


In [10]:
if len(gaps) and gaps['missing_dates'].max() > 0:
    worst = gaps.iloc[0][KEY_COLS].to_dict()
    print(f'Worst-affected key: {worst}')
    examples = ft.quality.gap_examples(df, spec, worst, max_examples=15)
    display(examples)

    # Distribution of missing weekdays — reveals structural patterns
    weekday_counts = examples['weekday'].value_counts()
    print('\nWeekday distribution of missing dates:')
    print(weekday_counts)
else:
    print('No time gaps detected — series are densely sampled at the spec frequency.')


Worst-affected key: {'unique_id': 2017}


,missing_date,weekday
0,2020-09-05,Saturday
1,2020-09-06,Sunday
2,2020-09-07,Monday
3,2020-09-08,Tuesday
4,2020-09-09,Wednesday
5,2020-09-10,Thursday
6,2020-09-11,Friday
7,2020-09-12,Saturday
8,2020-09-13,Sunday
9,2020-09-14,Monday



Weekday distribution of missing dates:
weekday
Saturday     3
Sunday       2
Monday       2
Tuesday      2
Wednesday    2
Thursday     2
Friday       2
Name: count, dtype: int64


>  **Reading the weekday counts.** If the missing dates are mostly
> Saturday/Sunday, the gap is probably structural (the business is
> closed). Treating that as a gap to "fill" with imputation would inject
> spurious demand into the series. Filling with **zero** is the right
> answer there. We'll do that in Notebook 5.


## 4.4 Duplicate (key, date) rows

Two rows with the same forecast key *and* the same date make the series
ambiguous: which one is the "true" observation? Models can't learn from
inconsistent labels, and downstream pivots/joins will silently inflate
totals.


In [11]:
dups = ft.quality.detect_duplicates(df, spec)
print(f'{len(dups):,} duplicate rows on (key + date).')
if len(dups):
    display(dups.head())
    print('\nResolve by either:')
    print('  • dropping (df = df.drop_duplicates(subset=key_cols+[date_col]))')
    print('  • aggregating (df = df.groupby(key_cols+[date_col]).agg({...}))')


0 duplicate rows on (key + date).


## 4.5 Outliers — three detectors, different blind spots

For multi-series datasets it's almost always wrong to compute outliers on
the *whole column*: a high-volume store has legitimately bigger numbers
than a low-volume store. The toolkit's detectors run **per series**.

| Method          | Statistic    | Strength                       | Weakness                    |
|-----------------|--------------|--------------------------------|-----------------------------|
| **IQR (Tukey)** | Quartiles    | Robust to a few extremes       | Misses milder anomalies     |
| **Z-score**     | Mean / std   | Easy to interpret              | Mean+std distorted by outliers themselves |
| **Modified Z**  | Median / MAD | Most robust on skewed sales    | Threshold (3.5) less familiar |

**Recommendation for sales/demand data: start with IQR or modified
Z-score. Plain Z-score is often too lenient on heavy-tailed series.**


In [12]:
out_iqr = ft.quality.outlier_summary(df, spec, method='iqr')
out_zsc = ft.quality.outlier_summary(df, spec, method='zscore')
out_mod = ft.quality.outlier_summary(df, spec, method='modified_zscore')

import pandas as pd
compare = pd.DataFrame({
    'iqr_outliers'           : out_iqr['n_outliers'].sum(),
    'iqr_pct'                : round(out_iqr['n_outliers'].sum() / out_iqr['n_obs'].sum() * 100, 2),
    'zscore_outliers'        : out_zsc['n_outliers'].sum(),
    'zscore_pct'             : round(out_zsc['n_outliers'].sum() / out_zsc['n_obs'].sum() * 100, 2),
    'modified_z_outliers'    : out_mod['n_outliers'].sum(),
    'modified_z_pct'         : round(out_mod['n_outliers'].sum() / out_mod['n_obs'].sum() * 100, 2),
}, index=['count']).T
compare.columns = ['value']
compare


,value
iqr_outliers,165345.00
iqr_pct,4.08
zscore_outliers,60953.00
zscore_pct,1.50
modified_z_outliers,119253.00
modified_z_pct,2.94


### Top series by outlier count


In [13]:
out_iqr.head(10)


,unique_id,n_obs,n_outliers,outlier_pct
0,2300,1412,459,32.51
1,2079,1412,410,29.04
2,2552,1412,357,25.28
3,1385,1412,303,21.46
4,5149,1412,271,19.19
5,2710,1144,263,22.99
6,4042,1412,255,18.06
7,2708,1102,254,23.05
8,2415,1193,229,19.20
9,3259,1412,223,15.79


### Visualise outliers on one example series

The plot below shows an actual series with IQR-flagged outliers in red.
Eyeballing this is essential — sometimes a "spike" is a Black-Friday or
promo event that you want the model to learn, **not** to remove.


In [14]:
worst_outlier_key = out_iqr.iloc[0][KEY_COLS].to_dict()
sub = ft.data_io.get_series(df, spec, worst_outlier_key).sort_values(DATE_COL)
flags = ft.quality.detect_outliers_iqr(sub[TARGET_COL])
fig = ft.plotting.plot_outliers_overlay(
    sub[TARGET_COL].reset_index(drop=True),
    flags.reset_index(drop=True),
    dates=sub[DATE_COL].reset_index(drop=True),
    title=f'IQR outliers — {worst_outlier_key}'
)
fig.show()


## 4.6 Sign and zero patterns of the target

A few quick checks that catch silly data bugs early:

- **Negative values** in a non-negative quantity (sales, demand) are
  almost always returns mis-coded as negative *gross* sales. Investigate.
- **Very high zero rate** flags an *intermittent* dataset and may also
  hint that gaps were already pre-filled with zeros at the source.


In [15]:
signs = ft.quality.target_sign_check(df, spec)
for k, v in signs.items():
    print(f'  {k:>20s} : {v}')


               n_total : 4054440
            n_negative : 0
                n_zero : 49277
            n_positive : 4005163
          pct_negative : 0.0
              pct_zero : 1.22


## 4.7 Are "static" features actually static?

Columns we declared as `STATIC_COLS` (e.g. `store_type`, `category`)
should have **one value per forecast key**. If they don't, either the
declaration is wrong or the data has changed mid-stream.

Skipped automatically if you didn't declare any static columns.


In [16]:
if STATIC_COLS:
    inconsistent = ft.quality.static_feature_consistency(df, spec)
    if len(inconsistent):
        print(f'  {len(inconsistent):,} keys have non-static "static" columns:')
        display(inconsistent.head(10))
    else:
        print(' All declared static columns are consistent within each key.')
else:
    print('No STATIC_COLS declared — skipping.')


  8 keys have non-static "static" columns:


,column,n_keys,n_inconsistent_keys,inconsistent_pct
0,warehouse,5390,0,0.0
1,product_unique_id,5390,0,0.0
2,name,5390,0,0.0
3,L1_category_name_en,5390,0,0.0
4,L2_category_name_en,5390,0,0.0
5,L3_category_name_en,5390,0,0.0
6,L4_category_name_en,5390,0,0.0
7,country,5390,0,0.0


## 4.8 Take-aways

| Check                         | What good looks like                                  |
|-------------------------------|-------------------------------------------------------|
| Missing value % per column    | Target < 5%; static cols 0%                           |
| Missing % per series          | No series above 50% missing                           |
| Time gaps                     | Either zero, or **structural and explainable**        |
| Duplicates on (key, date)     | Zero. Always.                                         |
| Outlier rate                  | Single-digit % per series; spikes recognisable        |
| Negative target values        | Zero, unless the domain explicitly allows them        |
| Static-column consistency     | All clean                                             |

**Document every issue you find here in a short data-quality report**
before moving on. The fixes happen in the next notebook.

Next: **Notebook 5 — Data Preprocessing** — gap-filling, imputation,
ffill/bfill of static columns, and outlier treatment.
